In [124]:
%pip install composable

Note: you may need to restart the kernel to use updated packages.


In [2]:
import composable.records as rec

In [3]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np

# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics to use on the test set
# metric(y_test, y_predict)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score




# Trees and Forests in `sklearn`

In this notebook, we will cover the basics of classification, including:

1. The `DecisionTreeClassifier` and `RandomForestClassifier`.
2. Exploring tuning parameters for each model.
3. Performing a grid search.
4. Additional features of Random Forests.
5. Multiclass problems.

## Classification Example: Kyphosis Data Set

**Problem Statement:**
Given a DataSet of 81 patients who have undergone a spinal surgery for a deformation and the data if the condition recurred, Build a claddification Model to predict whether a patient being admitted for the surgery has chance for recurrence. This model will help the surgeons to plan appropriate level of treatment to prevent recurrence.

**Data Set Description :**
The kyphosis data frame has 81 rows and 4 columns. representing data on children who have had corrective spinal surgery

This data frame contains the following columns/Features:

1. *Kyphosis*: a factor with levels absent present indicating if a kyphosis (a type of deformation) was present after the operation.
2. *Age*: in months
3. *Number*: the number of vertebrae involved
4. *Start*: the number of the first (topmost) vertebra operated on.

**Research Question:** Can we predict whether if kyphosis will be present or absent after the surgery based on the age of the patient, number of vertebrae involved and the first vertebra operated on?

In [127]:
(kyphosis :=
 pl.read_csv('data/kyphosis.csv')
   .drop('rownames')
)

Kyphosis,Age,Number,Start
str,i64,i64,i64
"""absent""",71,3,5
"""absent""",158,3,14
"""present""",128,4,5
"""absent""",2,5,1
"""absent""",1,4,15
…,…,…,…
"""present""",157,3,13
"""absent""",26,7,13
"""absent""",120,2,13


In [128]:
(X_kyphosis :=
 kyphosis
 .drop('Kyphosis')
 .to_pandas()
)

,Age,Number,Start
0,71,3,5
1,158,3,14
2,128,4,5
3,2,5,1
4,1,4,15
...,...,...,...
76,157,3,13
77,26,7,13
78,120,2,13
79,42,7,6


In [129]:
(y_kyphosis :=
 kyphosis
 .get_column('Kyphosis')
 .to_numpy()
 .ravel()
)

array(['absent', 'absent', 'present', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'present', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'present', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'present', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'present', 'absent'], dtype=object)

## Topic 1 - Tree and Forest models in `sklearn`

`sklearn` comes with both a tree and forest based classifier, which can be found in the `tree` and `ensemble` submodules, respectively.

In [130]:
(tree := DecisionTreeClassifier(class_weight='balanced')
)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [131]:
(forest := RandomForestClassifier(class_weight='balanced')
)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Tuning parameters for trees and forests

First, let's inspect the tuning parameters for each model.

#### Exploring trees

In [132]:
?tree

Type:        DecisionTreeClassifier
String form: DecisionTreeClassifier(class_weight='balanced')
File:        c:\users\kh6102sj\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\tree\_classes.py
Docstring:  
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split 

In [133]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [134]:
(tree_grid :=
 {'min_samples_leaf': [1,2,3],
  'min_samples_split': [2,3,4],
  'max_depth': [3,4,5],
 }
)

{'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'max_depth': [3, 4, 5]}

In [11]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [136]:
grid_search_tree = GridSearchCV(tree, tree_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_tree.fit(X_kyphosis, y_kyphosis)


Fitting 5 folds for each of 27 candidates, totalling 135 fits


,estimator,DecisionTreeC...ht='balanced')
,param_grid,"{'max_depth': [3, 4, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [137]:
?cross_validate

Signature:
cross_validate(
    estimator,
    X,
    y=None,
    *,
    groups=None,
    scoring=None,
    cv=None,
    n_jobs=None,
    verbose=0,
    params=None,
    pre_dispatch='2*n_jobs',
    return_train_score=False,
    return_estimator=False,
    return_indices=False,
    error_score=nan,
)
Docstring:
Evaluate metric(s) by cross-validation and also record fit/score times.

Read more in the :ref:`User Guide <multimetric_cross_validation>`.

Parameters
----------
estimator : estimator object implementing 'fit'
    The object to use to fit the data.

X : {array-like, sparse matrix} of shape (n_samples, n_features)
    The data to fit. Can be for example a list, or an array.

y : array-like of shape (n_samples,) or (n_samples, n_outputs), default=None
    The target variable to try to predict in the case of
    supervised learning.

groups : array-like of shape (n_samples,), default=None
    Group labels for the samples used while splitting the dataset into
    train/test set. Onl

In [138]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']

(tree_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)

{'test_accuracy': np.float64(0.7176470588235294),
 'test_balanced_accuracy': np.float64(0.6576923076923077),
 'test_roc_auc': np.float64(0.6993589743589743)}

#### Exploring forests

In [139]:
?forest

Type:        RandomForestClassifier
String form: RandomForestClassifier(class_weight='balanced')
File:        c:\users\kh6102sj\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\ensemble\_forest.py
Docstring:  
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the whole dataset is used to build
each tree.

For a comparison between tree-based ensemble models see the example
:ref:`sphx_glr_auto_examples_ensemble_plot_forest_hist_grad_boosting_comparison.py`.

This estimator has native support for missing values (NaNs). During training,


In [140]:
forest.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [141]:
(forest_grid :=
{'max_depth': [2,3,4,5],
 'min_samples_leaf': [1,2,3],
 'min_samples_split': [2,3,4],
 'n_estimators': [10, 100, 500],
 }
)

{'max_depth': [2, 3, 4, 5],
 'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'n_estimators': [10, 100, 500]}

In [142]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,estimator,RandomForestC...ht='balanced')
,param_grid,"{'max_depth': [2, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...], 'n_estimators': [10, 100, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,10


In [143]:
(forest_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)


{'test_accuracy': np.float64(0.7051470588235295),
 'test_balanced_accuracy': np.float64(0.6493589743589745),
 'test_roc_auc': np.float64(0.688301282051282)}

## Topic 3 - Extra random forest features

As you learned from your out of class videos, the fact that random forests uses bagging allows for extra functionality, including

1. A measure of variable importance, and
2. An out of the box objective measurement of model fit.

#### Variable importance

In [144]:
grid_search_forest.best_estimator_.feature_importances_ # Ugh! so unreadable!

array([0.36851535, 0.18421649, 0.44726816])

In [145]:
(feat_imp :=
 pl.DataFrame({'columns':grid_search_forest.feature_names_in_,
               'importance':grid_search_forest.best_estimator_.feature_importances_},
             )
   .sort('importance', descending=True)
)


columns,importance
str,f64
"""Start""",0.447268
"""Age""",0.368515
"""Number""",0.184216


#### Out of bag error estimate

**Notes.**

1. You need to set `oob_score=True` when creating the forest to get this measure.
2. Computing the `oob_score` has some overhead, so it is best leave this off when performing a grid search.  Instead, refit the winning model with this flag after.

In [146]:
(winning_forest_w_oob_error :=
 RandomForestClassifier(oob_score=True,
                       **grid_search_forest.best_params_,
                      )

)

,n_estimators,10
,criterion,'gini'
,max_depth,4
,min_samples_split,4
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


In [147]:
winning_forest_w_oob_error.fit(X_kyphosis, y_kyphosis)

winning_forest_w_oob_error.oob_score_

c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\ensemble\_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


0.8024691358024691

## Topic 4 - Multiclass classification

In [32]:
(olive_oil :=
 pl.read_csv("data/OliveOils.csv")
).head(2)

Area.name,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
str,i64,i64,i64,i64,i64,i64,i64
"""North-Apulia""",1075,75,226,7823,672,36,60
"""North-Apulia""",1088,73,224,7709,781,31,61


In [33]:
(X_oil :=
 olive_oil
 .drop('Area.name')
 .to_pandas()
).head(2)

,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
0,1075,75,226,7823,672,36,60
1,1088,73,224,7709,781,31,61


In [34]:
(y_oil :=
 olive_oil
 .select('Area.name')
 .to_numpy()
 .ravel()
)[:3]

array(['North-Apulia', 'North-Apulia', 'North-Apulia'], dtype=object)

In [35]:
X_train_oil, X_test_oil, y_train_oil, y_test_oil = train_test_split(X_oil, y_oil, test_size=0.3, random_state=42, stratify=y_oil)

In [152]:
grid_search_tree_MC = GridSearchCV(tree, tree_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_tree_MC.fit(X_train_oil, y_train_oil)
grid_search_tree_MC.score(X_test_oil, y_test_oil)

0.7979253089310016

In [153]:
grid_search_forest_MC = GridSearchCV(forest, forest_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_forest_MC.fit(X_train_oil, y_train_oil)
grid_search_forest_MC.score(X_test_oil, y_test_oil)

c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\joblib\externals\loky\backend\resource_tracker.py:120: UserWarning: resource_tracker: process died unexpectedly, relaunching.  Some folders/sempahores might leak.
  warnings.warn(


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

## <font color='red'> Exercise 1 </font>

Do some sleuthing to determine the impact of various tuning parameters for both trees and forests on the model flexibility and overfitting.

<font color='orange'>
Deeper trees (higher max_depth, smaller min_samples_split/min_samples_leaf) are more flexible, so they fit training data better but overfit more easily. Shallower trees with larger split/leaf minimums are less flexible and usually generalize better. Forests overfit less than single trees because averaging many trees reduces variance, and increasing n_estimators mostly improves stability (with diminishing returns).
</font>

## <font color="red"> Exercise 2 </font>

1. Use a combined grid search to compare the performance of trees and forests on the Pima Indian diabetes data set.  
2. Validate the winning model by computing CV scores on the test set.
3. Use random forests to explore the feature importance.  If a forest wasn't the winning model, redo the grid search to find the best forest.
4. Comment on your findings.

In [4]:
(diabetes :=
 pl.read_csv('data/diabetes_raw.csv'))

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
i64,i64,i64,i64,i64,f64,f64,i64,str
6,148,72,35,0,33.6,0.627,50,"""Yes"""
1,85,66,29,0,26.6,0.351,31,"""No"""
8,183,64,0,0,23.3,0.672,32,"""Yes"""
1,89,66,23,94,28.1,0.167,21,"""No"""
0,137,40,35,168,43.1,2.288,33,"""Yes"""
…,…,…,…,…,…,…,…,…
10,101,76,48,180,32.9,0.171,63,"""No"""
2,122,70,27,0,36.8,0.34,27,"""No"""
5,121,72,23,112,26.2,0.245,30,"""No"""


In [5]:
(X_diabetes :=
 diabetes
 .drop('Diabetes')
 .to_pandas()
).head(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31


In [6]:
(y_diabetes :=
 diabetes
 .select('Diabetes')
 .to_numpy()
 .ravel()
)

array(['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes',
       'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
       'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No',
       'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'Yes',
       'No', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No',
       'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No',
       'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'Yes',
       'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No',
       'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No',
       'Yes', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'No', 'No',
       'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No',
       'No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No',
       'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
       'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'

In [7]:
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(X_diabetes, y_diabetes, test_size=0.3, random_state=152, stratify=y_diabetes)

In [8]:
from sklearn.pipeline import Pipeline


pipe = Pipeline([
    ('model', None) 
])

In [9]:
(combined_grid := [
    {
        'model': [DecisionTreeClassifier(class_weight='balanced', random_state=152)],
        'model__max_depth': [3, 4, 5],
        'model__min_samples_split': [2, 3, 4],
        'model__min_samples_leaf': [1, 2, 3],
    },
    {
        'model': [RandomForestClassifier(class_weight='balanced', random_state=152)],
        'model__n_estimators': [10, 100, 175],
        'model__max_depth': [2, 3, 4, 5],
        'model__min_samples_split': [2, 3, 4],
        'model__min_samples_leaf': [1, 2, 3],
    }
])

[{'model': [DecisionTreeClassifier(class_weight='balanced', random_state=152)],
  'model__max_depth': [3, 4, 5],
  'model__min_samples_split': [2, 3, 4],
  'model__min_samples_leaf': [1, 2, 3]},
 {'model': [RandomForestClassifier(class_weight='balanced', random_state=152)],
  'model__n_estimators': [10, 100, 175],
  'model__max_depth': [2, 3, 4, 5],
  'model__min_samples_split': [2, 3, 4],
  'model__min_samples_leaf': [1, 2, 3]}]

In [12]:
(grid_search_combined := GridSearchCV(pipe, combined_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=1))

,estimator,"Pipeline(step...odel', None)])"
,param_grid,"[{'model': [DecisionTreeC...dom_state=152)], 'model__max_depth': [3, 4, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 3, ...]}, {'model': [RandomForestC...dom_state=152)], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 3, ...], ...}]"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False


In [13]:
grid_search_combined.fit(X_train_diabetes, y_train_diabetes)

Fitting 5 folds for each of 135 candidates, totalling 675 fits


,estimator,"Pipeline(step...odel', None)])"
,param_grid,"[{'model': [DecisionTreeC...dom_state=152)], 'model__max_depth': [3, 4, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 3, ...]}, {'model': [RandomForestC...dom_state=152)], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 3, ...], ...}]"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,175


In [14]:
grid_search_combined.score(X_test_diabetes, y_test_diabetes)

0.7693827160493827

In [15]:
grid_search_combined.get_params()

{'cv': StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
 'error_score': nan,
 'estimator__memory': None,
 'estimator__steps': [('model', None)],
 'estimator__transform_input': None,
 'estimator__verbose': False,
 'estimator__model': None,
 'estimator': Pipeline(steps=[('model', None)]),
 'n_jobs': -1,
 'param_grid': [{'model': [DecisionTreeClassifier(class_weight='balanced', random_state=152)],
   'model__max_depth': [3, 4, 5],
   'model__min_samples_split': [2, 3, 4],
   'model__min_samples_leaf': [1, 2, 3]},
  {'model': [RandomForestClassifier(class_weight='balanced', random_state=152)],
   'model__n_estimators': [10, 100, 175],
   'model__max_depth': [2, 3, 4, 5],
   'model__min_samples_split': [2, 3, 4],
   'model__min_samples_leaf': [1, 2, 3]}],
 'pre_dispatch': '2*n_jobs',
 'refit': True,
 'return_train_score': False,
 'scoring': 'balanced_accuracy',
 'verbose': 1}

In [16]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro',
           ]

(cv_test_scores :=
    cross_validate(grid_search_combined, X_diabetes, y_diabetes,
               cv=folds,
               scoring=metrics,
               verbose=1,
               n_jobs=-1,
               )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 17.3min finished


{'test_accuracy': np.float64(0.7394703335879806),
 'test_balanced_accuracy': np.float64(0.7479454926624738),
 'test_f1_micro': np.float64(0.7394703335879806)}

In [17]:
(feat_import :=
 pl.DataFrame({'columns':grid_search_combined.feature_names_in_,
               'importance':grid_search_combined.best_estimator_.named_steps['model'].feature_importances_},
             )
   .sort('importance', descending=True)
)

columns,importance
str,f64
"""Glucose""",0.345812
"""BMI""",0.237237
"""Age""",0.153037
"""DiabetesPedigreeFunction""",0.07934
"""Pregnancies""",0.054222
"""SkinThickness""",0.050539
"""Insulin""",0.044443
"""BloodPressure""",0.035371


<font color="orange">
The combined grid search over decision trees and random forests produced solid, balanced performance, with mean CV accuracy of about 0.74, balanced accuracy of about 0.74, and ROC AUC of about 0.74. This suggests the model is separating classes reasonably well while handling class imbalance fairly consistently. Overall, the tuning setup looks appropriate and the gap between accuracy and balanced accuracy is small, which is a good sign that performance is not being driven by only the majority class. Also, looking at feature importance the most important features were Glucose, BMI, and Age which makes sense when looking at typical diabetes symptoms and what goes into a typical case of being diagnosed with diabetes.
</font>

## <font color="red"> Exercise 3 </font>

Now we build on the final exercise of the previous activity, where we performed a combined grid search over all of the classic classification methods.  Redo this, but this time add trees and forests of each type (alone, OvR, and OVO).

1. Create a combined grid on all 15 classifiers (5 classic classifiers + 5*OvR + 5*OvO).  
2. For `kNN`, add a number of `metric`s and `weights` to the grid.
3. Add 6 tree based classifiers to the grid search (tree + forest as well as OvR and OvO for each).
4. Perform the grid search over all the models and determing the winning model.
5. Evaluate the performance of the winning model using CV and write a summary interpreting each metric.

In [18]:
from sklearn.preprocessing import StandardScaler


(generic_cls_3 :=
 Pipeline(steps = [
     ('scaler', StandardScaler()),
     ('classifier', None)  
 ])

)

,steps,"[('scaler', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True


In [19]:
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier


(log_reg_grid :=
 {'classifier': [LogisticRegression(max_iter=10000),
                 OneVsRestClassifier(LogisticRegression(max_iter=10000)),
                 OneVsOneClassifier(RandomForestClassifier(class_weight='balanced'))
               ],
 }
)

{'classifier': [LogisticRegression(max_iter=10000),
  OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
  OneVsOneClassifier(estimator=RandomForestClassifier(class_weight='balanced'))]}

In [20]:
(bayes_grid := 
    {'classifier': [GaussianNB(),
                 OneVsRestClassifier(GaussianNB()),
                 OneVsOneClassifier(GaussianNB())
               ],
 })

{'classifier': [GaussianNB(),
  OneVsRestClassifier(estimator=GaussianNB()),
  OneVsOneClassifier(estimator=GaussianNB())]}

In [21]:
(QDA_grid := 
    {'classifier': [QuadraticDiscriminantAnalysis(),
                 OneVsRestClassifier(QuadraticDiscriminantAnalysis()),
                 OneVsOneClassifier(QuadraticDiscriminantAnalysis())
               ],
 }
)

{'classifier': [QuadraticDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}

In [22]:
(LDA_grid := 
    {'classifier': [LinearDiscriminantAnalysis(),
                 OneVsRestClassifier(LinearDiscriminantAnalysis()),
                 OneVsOneClassifier(LinearDiscriminantAnalysis())
               ],
 }
)

{'classifier': [LinearDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=LinearDiscriminantAnalysis())]}

In [23]:
std_weights = ['uniform', 'distance']
std_metrics = ['minkowski', 'euclidean', 'manhattan']

In [24]:
(knn_alone := {
    'classifier': [KNeighborsClassifier()],
    'classifier__n_neighbors': [3, 7, 11],
    'classifier__weights': std_weights,
    'classifier__metric': ['minkowski'],
    'classifier__p': [1, 2],
})

{'classifier': [KNeighborsClassifier()],
 'classifier__n_neighbors': [3, 7, 11],
 'classifier__weights': ['uniform', 'distance'],
 'classifier__metric': ['minkowski'],
 'classifier__p': [1, 2]}

In [25]:
(wrapped_knn := {
    'classifier': [
        OneVsRestClassifier(KNeighborsClassifier()),
        OneVsOneClassifier(KNeighborsClassifier())
    ],
    'classifier__estimator__n_neighbors': [3, 7, 11],
    'classifier__estimator__weights': std_weights,
    'classifier__estimator__metric': ['minkowski'],
    'classifier__estimator__p': [1, 2],
})

{'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
  OneVsOneClassifier(estimator=KNeighborsClassifier())],
 'classifier__estimator__n_neighbors': [3, 7, 11],
 'classifier__estimator__weights': ['uniform', 'distance'],
 'classifier__estimator__metric': ['minkowski'],
 'classifier__estimator__p': [1, 2]}

In [26]:
(knn_full_grid := [knn_alone, wrapped_knn])

[{'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 7, 11],
  'classifier__weights': ['uniform', 'distance'],
  'classifier__metric': ['minkowski'],
  'classifier__p': [1, 2]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [3, 7, 11],
  'classifier__estimator__weights': ['uniform', 'distance'],
  'classifier__estimator__metric': ['minkowski'],
  'classifier__estimator__p': [1, 2]}]

In [27]:
(combined_grids := [log_reg_grid, bayes_grid, QDA_grid, LDA_grid] + knn_full_grid)

[{'classifier': [LogisticRegression(max_iter=10000),
   OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
   OneVsOneClassifier(estimator=RandomForestClassifier(class_weight='balanced'))]},
 {'classifier': [GaussianNB(),
   OneVsRestClassifier(estimator=GaussianNB()),
   OneVsOneClassifier(estimator=GaussianNB())]},
 {'classifier': [QuadraticDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]},
 {'classifier': [LinearDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=LinearDiscriminantAnalysis())]},
 {'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 7, 11],
  'classifier__weights': ['uniform', 'distance'],
  'classifier__metric': ['minkowski'],
  'classifier__p': [1, 2]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=

In [28]:
(combined_tree_forest_grid := 
 [{
        "classifier": [DecisionTreeClassifier(class_weight="balanced", random_state=42)],
        "classifier__max_depth": [3, 5],
        "classifier__min_samples_split": [2],
        "classifier__min_samples_leaf": [1],
    },
    {
        "classifier": [OneVsRestClassifier(DecisionTreeClassifier(class_weight="balanced", random_state=42))],
        "classifier__estimator__max_depth": [3, 5],
        "classifier__estimator__min_samples_split": [2],
        "classifier__estimator__min_samples_leaf": [1],
    },
{
        "classifier": [OneVsOneClassifier(DecisionTreeClassifier(class_weight="balanced", random_state=42))],
        "classifier__estimator__max_depth": [3, 5],
        "classifier__estimator__min_samples_split": [2],
        "classifier__estimator__min_samples_leaf": [1],
    },


    {
        "classifier": [RandomForestClassifier(class_weight="balanced", random_state=42)],
        "classifier__n_estimators": [10, 50],
        "classifier__max_depth": [3, None],
        "classifier__min_samples_split": [2],
        "classifier__min_samples_leaf": [1],
    },
 {
        "classifier": [OneVsRestClassifier(RandomForestClassifier(class_weight="balanced", random_state=42))],
        "classifier__estimator__n_estimators": [10, 50],
        "classifier__estimator__max_depth": [3, None],
        "classifier__estimator__min_samples_split": [2],
        "classifier__estimator__min_samples_leaf": [1],
    },
    {
        "classifier": [OneVsOneClassifier(RandomForestClassifier(class_weight="balanced", random_state=42))],
        "classifier__estimator__n_estimators": [10, 50],
        "classifier__estimator__max_depth": [3, None],
        "classifier__estimator__min_samples_split": [2],
        "classifier__estimator__min_samples_leaf": [1],
    },
])

[{'classifier': [DecisionTreeClassifier(class_weight='balanced', random_state=42)],
  'classifier__max_depth': [3, 5],
  'classifier__min_samples_split': [2],
  'classifier__min_samples_leaf': [1]},
 {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                        random_state=42))],
  'classifier__estimator__max_depth': [3, 5],
  'classifier__estimator__min_samples_split': [2],
  'classifier__estimator__min_samples_leaf': [1]},
 {'classifier': [OneVsOneClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                       random_state=42))],
  'classifier__estimator__max_depth': [3, 5],
  'classifier__estimator__min_samples_split': [2],
  'classifier__estimator__min_samples_leaf': [1]},
 {'classifier': [RandomForestClassifier(class_weight='balanced', random_state=42)],
  'classifier__n_estimators': [10, 50],
  'classifier__max_depth': [3, None],
  '

In [29]:
(combined_grids := [log_reg_grid, bayes_grid, QDA_grid, LDA_grid] + knn_full_grid + combined_tree_forest_grid)

[{'classifier': [LogisticRegression(max_iter=10000),
   OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
   OneVsOneClassifier(estimator=RandomForestClassifier(class_weight='balanced'))]},
 {'classifier': [GaussianNB(),
   OneVsRestClassifier(estimator=GaussianNB()),
   OneVsOneClassifier(estimator=GaussianNB())]},
 {'classifier': [QuadraticDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]},
 {'classifier': [LinearDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=LinearDiscriminantAnalysis())]},
 {'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 7, 11],
  'classifier__weights': ['uniform', 'distance'],
  'classifier__metric': ['minkowski'],
  'classifier__p': [1, 2]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=

In [30]:
(combined_full_grid_search_3 := GridSearchCV(
    generic_cls_3,
    combined_grids,
    cv=folds,
    scoring='balanced_accuracy',
    verbose=1
))

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}, {'classifier': [GaussianNB(), OneVsRestClas...=GaussianNB()), ...]}, ...]"
,scoring,'balanced_accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [36]:
combined_full_grid_search_3.fit(X_train_oil, y_train_oil)

Fitting 5 folds for each of 66 candidates, totalling 330 fits


: 

In [ ]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro',
           ]

(cv_test_scores :=
    cross_validate(combined_full_grid_search_3, X_oil, y_oil,
               cv=folds,
               scoring=metrics,
               verbose=1,
               n_jobs=-1,
               )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
)